# Member 3 — K-Nearest Neighbors (KNN)
## FIFA Player Position Prediction

---

**Name:** FIFA 22 Complete Player Dataset  

**Source:** https://www.kaggle.com/datasets/stefanoleone992/fifa-22-complete-player-dataset  

**Size:** ~19,000 players, 100+ attributes  

**Context:** FIFA 22 is a football simulation game. Player statistics reflect real-world player abilities.  

**Algorithm:** K-Nearest Neighbors  

**Features used:** age, height_cm, weight_kg, overall, potential, pace, shooting, passing, dribbling, defending, physic  

**Task:** Predict if a player is ATTACKER, MIDFIELDER, or DEFENDER

---

## 1. Introduction to K-Nearest Neighbors

**KNN** is one of the simplest ML algorithms. Here's the idea:

When predicting the position of a NEW player, KNN:
1. Looks at the K most SIMILAR players it has already seen
2. Takes a majority VOTE from their positions
3. Assigns the most common position to the new player

Example: If K=5, and 3 of the 5 most similar players are Attackers, it predicts ATTACKER.

**Why KNN for this task?**
- Very intuitive: similar stats = similar position
- No training phase (just stores data)
- Works well for classification problems
- Easy to tune (just change K)

## 2. Import libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# KNN from scikit-learn
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    auc,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.preprocessing import label_binarize

print('Libraries imported!')

Libraries imported!


## 3. Load the preprocessed data

In [2]:
X_train = np.load('X_train.npy')
X_test  = np.load('X_test.npy')
y_train = np.load('y_train.npy')
y_test  = np.load('y_test.npy')
class_names = np.load('class_names.npy', allow_pickle=True)

print('Data loaded!')
print(f'Training: {len(X_train)} | Testing: {len(X_test)} | Classes: {list(class_names)}')

Data loaded!
Training: 15345 | Testing: 3422 | Classes: ['ATTACKER', 'DEFENDER', 'MIDFIELDER']


## 4. Find the best K value

The most important hyperparameter in KNN is K (how many neighbors to look at).
We test K from 1 to 20 and pick the best one.

In [ ]:
# Test different values of K
k_values = list(range(1, 21))  # K from 1 to 20
accuracies = []

print('Testing different K values...')
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    knn.fit(X_train, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test))
    accuracies.append(acc * 100)
    print(f'  K={k:2d} → Accuracy: {acc*100:.2f}%')

# Best K
best_k = k_values[np.argmax(accuracies)]
print(f'\nBest K = {best_k} with accuracy = {max(accuracies):.2f}%')

In [ ]:
# Plot K vs Accuracy
plt.figure(figsize=(10, 5))
plt.plot(k_values, accuracies, marker='o', color='purple', linewidth=2)
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.title('KNN — Accuracy vs K Value')
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('member3_k_vs_accuracy.png')
plt.show()

## 5. Train final model with best K

In [ ]:
# Train with the best K found above
knn_model = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
knn_model.fit(X_train, y_train)

print(f'KNN model trained with K={best_k}')

## 6. Evaluate the model

In [ ]:
y_pred = knn_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'=== KNN RESULTS ===')
print(f'Best K     : {best_k}')
print(f'Accuracy   : {accuracy * 100:.2f}%')
print()
print('Detailed Report:')
print(classification_report(y_test, y_pred, target_names=class_names))

## 7. Evaluation Matrics

In [ ]:
# ── 1. Precision, Recall & F1-Score ──────────────────────────────────────────
precision_macro    = precision_score(y_test, y_pred, average='macro',    zero_division=0)
precision_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall_macro       = recall_score   (y_test, y_pred, average='macro',    zero_division=0)
recall_weighted    = recall_score   (y_test, y_pred, average='weighted', zero_division=0)
f1_macro           = f1_score       (y_test, y_pred, average='macro',    zero_division=0)
f1_weighted        = f1_score       (y_test, y_pred, average='weighted', zero_division=0)

print('=== PRECISION, RECALL & F1-SCORE ===')
print(f'  Precision  (macro)    : {precision_macro:.4f}')
print(f'  Precision  (weighted) : {precision_weighted:.4f}')
print(f'  Recall     (macro)    : {recall_macro:.4f}')
print(f'  Recall     (weighted) : {recall_weighted:.4f}')
print(f'  F1-Score   (macro)    : {f1_macro:.4f}')
print(f'  F1-Score   (weighted) : {f1_weighted:.4f}')
print()

# Per-class breakdown
precision_per = precision_score(y_test, y_pred, average=None, zero_division=0)
recall_per    = recall_score   (y_test, y_pred, average=None, zero_division=0)
f1_per        = f1_score       (y_test, y_pred, average=None, zero_division=0)

print(f'{"Class":<20} {"Precision":>10} {"Recall":>10} {"F1-Score":>10}')
print('-' * 52)
for i, cls in enumerate(class_names):
    print(f'{cls:<20} {precision_per[i]:>10.4f} {recall_per[i]:>10.4f} {f1_per[i]:>10.4f}')

In [ ]:
# ── 2. ROC-AUC (One-vs-Rest for multi-class) ──────────────────────────────────
# KNN probability scores
y_prob = knn_model.predict_proba(X_test)

# Binarize labels for one-vs-rest ROC
classes = np.unique(y_train)
y_test_bin = label_binarize(y_test, classes=classes)

roc_auc_macro    = roc_auc_score(y_test_bin, y_prob, multi_class='ovr', average='macro')
roc_auc_weighted = roc_auc_score(y_test_bin, y_prob, multi_class='ovr', average='weighted')

print('=== ROC-AUC SCORE (One-vs-Rest) ===')
print(f'  ROC-AUC (macro)    : {roc_auc_macro:.4f}')
print(f'  ROC-AUC (weighted) : {roc_auc_weighted:.4f}')

# Plot ROC curve for each class
plt.figure(figsize=(10, 7))
colors = plt.cm.Set2(np.linspace(0, 1, len(classes)))

for i, (cls_idx, color) in enumerate(zip(classes, colors)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    roc_auc_cls = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'{class_names[i]} (AUC = {roc_auc_cls:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'KNN (K={best_k}) — ROC Curve (One-vs-Rest per Class)')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('member3_roc_curve.png')
plt.show()
print('ROC curve saved!')

In [ ]:
# ── 3. Confusion Matrix with TP/FP/TN/FN ─────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title(f'KNN (K={best_k}) — Confusion Matrix')
plt.ylabel('Actual Position')
plt.xlabel('Predicted Position')
plt.tight_layout()
plt.savefig('member3_confusion_matrix.png')
plt.show()
print('Confusion matrix saved!')

# TP / FP / TN / FN breakdown per class
print()
print(f'{"Class":<20} {"TP":>6} {"FP":>6} {"TN":>6} {"FN":>6}')
print('-' * 44)
for i, cls in enumerate(class_names):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - TP - FP - FN
    print(f'{cls:<20} {TP:>6} {FP:>6} {TN:>6} {FN:>6}')